# Sweep — Weight decay

Proxy fold 0 with gamma=1 fixed. 4 weight decay values x 5 epochs. Winner: 1e-4.


In [1]:
from google.colab import drive
drive.mount("/content/drive")
%run colab_init.py

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn matplotlib seaborn kaggle

from birdclef.colab import colab_prepare_training

colab_prepare_training(stage_tta=False)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
train.csv already at /content/train.csv
sample_submission.csv already at /content/sample_submission.csv
Baseline embeddings already staged at /content/embeddings_v2
TTA embeddings already staged at /content/embeddings_v2_TTA
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df_TTA, NUM_CLASSES, _ = prepare_tta_data()
PROXY_FOLD = 0
EPOCHS = 5


In [4]:
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import BirdDataset, BirdMoE, FocalLoss, competition_macro_roc_auc, seed_everything
from birdclef.paths import SWEEPS_DIR
from birdclef.sweep_plots import plot_weight_decay_sweep

SAVE_DIR = SWEEPS_DIR / "weight_decay"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

WINNING_GAMMA = 1.0
WEIGHT_DECAY_VALUES = [1e-5, 1e-4, 1e-3, 1e-2]
train_df = df_TTA[df_TTA["fold"] != PROXY_FOLD].reset_index(drop=True)
valid_df = df_TTA[df_TTA["fold"] == PROXY_FOLD].reset_index(drop=True)
train_loader = DataLoader(BirdDataset(train_df, is_tta=True), batch_size=64, shuffle=True, num_workers=0)
valid_loader = DataLoader(BirdDataset(valid_df, is_tta=True), batch_size=64, shuffle=False, num_workers=0)

sweep_results = {}
sweep_history = {}

for wd in WEIGHT_DECAY_VALUES:
    print(f"Testing weight_decay={wd}")
    seed_everything(42)
    model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)
    criterion = FocalLoss(gamma=WINNING_GAMMA)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=wd)
    best_auc = 0.0
    val_aucs = []

    for epoch in range(EPOCHS):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if torch.isnan(inputs).any():
                continue
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        val_aucs.append(current_auc)
        best_auc = max(best_auc, current_auc)
        print(f"  Epoch {epoch + 1}/{EPOCHS} | Val AUC: {current_auc:.4f}")

    sweep_results[wd] = best_auc
    sweep_history[wd] = {"val_aucs": val_aucs}
    print(f"Peak AUC for wd={wd}: {best_auc:.4f}")

best_wd = max(sweep_results, key=sweep_results.get)
print(f"Winner: wd={best_wd} (AUC={sweep_results[best_wd]:.4f})")
plot_weight_decay_sweep(sweep_results, sweep_history, str(SAVE_DIR))


100%|██████████| 7110/7110 [00:09<00:00, 752.39it/s]


Testing weight_decay=1e-05
  Epoch 1/5 | Val AUC: 0.9781
  Epoch 2/5 | Val AUC: 0.9769
  Epoch 3/5 | Val AUC: 0.9762
  Epoch 4/5 | Val AUC: 0.9756
  Epoch 5/5 | Val AUC: 0.9748
Peak AUC for wd=1e-05: 0.9781
Testing weight_decay=0.0001
  Epoch 1/5 | Val AUC: 0.9754
  Epoch 2/5 | Val AUC: 0.9752
  Epoch 3/5 | Val AUC: 0.9762
  Epoch 4/5 | Val AUC: 0.9775
  Epoch 5/5 | Val AUC: 0.9778
Peak AUC for wd=0.0001: 0.9778
Testing weight_decay=0.001
  Epoch 1/5 | Val AUC: 0.9751
  Epoch 2/5 | Val AUC: 0.9750
  Epoch 3/5 | Val AUC: 0.9749
  Epoch 4/5 | Val AUC: 0.9747
  Epoch 5/5 | Val AUC: 0.9761
Peak AUC for wd=0.001: 0.9761
Testing weight_decay=0.01
  Epoch 1/5 | Val AUC: 0.8837
  Epoch 2/5 | Val AUC: 0.8814
  Epoch 3/5 | Val AUC: 0.8786
  Epoch 4/5 | Val AUC: 0.8776
  Epoch 5/5 | Val AUC: 0.8794
Peak AUC for wd=0.01: 0.8837
Winner: wd=1e-05 (AUC=0.9781)
Weight decay sweep plots saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/sweeps/weight_decay
